# Introduction

Adversarial examples highlight how vulnerable computer vision models remain to small, carefully designed perturbations. In Adversarial Evasion Attacks on Computer Vision using SHAP Values, Mollard, Becker, and Roehrbein propose a white-box evasion attack that uses SHAP values instead of raw input gradients to identify the pixels that most strongly support or oppose a model’s prediction. The core idea is to move highly influential pixels toward a more neutral attribution range, which can reduce class confidence and trigger misclassification while keeping the perturbation visually subtle.

![https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcR5lNcUsVDU6d4HTnBdD1cSdTUwrEonP_f1SQ&s](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcR5lNcUsVDU6d4HTnBdD1cSdTUwrEonP_f1SQ&s)

This paper is especially relevant for a Kaggle notebook because it connects explainability and robustness. SHAP is usually used to interpret model decisions, but here it is also used to expose and exploit model weaknesses. The authors evaluate the approach across multiple datasets and model families, and compare it with an FGSM-based baseline. Their results suggest that SHAP-guided attacks are often more stable and can remain effective when gradients become less informative, although the advantage is not uniform across every dataset. A practical trade-off is computational cost: SHAP-based attacks are substantially heavier than gradient-based ones, especially for larger models or higher-resolution images.

[Link to original paper](https://arxiv.org/pdf/2601.10587)

Mollard, F., Becker, M., & Roehrbein, F. (2026). Adversarial Evasion Attacks on Computer Vision using SHAP Values. arXiv. doi:10.48550/arXiv.2601.10587.

In [1]:
import warnings
import gc

import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
plt.rcParams['font.size'] = 17
import matplotlib.image as mpimg
from PIL import Image

from tensorflow import concat, zeros, ones, stack, convert_to_tensor, reduce_mean, config
from tensorflow.math import log as tfLog
from tensorflow.random import set_seed, categorical, normal
set_seed(2024)
from tensorflow.keras.layers import Input, Dense, Conv2D, MaxPooling2D, AveragePooling2D, Reshape, Flatten, UpSampling2D, Conv2DTranspose, \
BatchNormalization, Dropout, concatenate, Add, Embedding
from tensorflow.keras import Model, backend
from tensorflow.keras.utils import plot_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import History
import kagglehub
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold

import shap

warnings.filterwarnings("ignore")

2026-03-18 18:04:51.520452: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773857091.796324      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773857091.871339      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773857092.525332      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773857092.525379      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773857092.525382      55 computation_placer.cc:177] computation placer alr

In [ ]:
data_path = r"/kaggle/input/datasets/frankmollard/animals-and-humans"
image_data = "imgs_dcwhBig.npy"
category_data = "categories_dcwhBig.npy"

In [ ]:
def giveIMG(row, shape=(64,64,3)):

    row = row*255
    img = row.reshape(shape).astype('uint8')

    return img

## Load Categories

In [ ]:
C = np.load(data_path + "/" + category_data)
print("Shape C:", C.shape)
numCategories = len(set(C))
print("Number of categories:", numCategories)

Now we have reduced the dataset to cats and dogs.

In [ ]:
C

In [ ]:
target = [i for i in C if i in ["cat", "dog"]]

In [ ]:
len(target)

1 if cat, else dog

In [ ]:
target = np.array([1 if i == "cat" else 0 for i in target])
target

## Load Images

In [ ]:
gc.collect()
images=np.load(data_path + "/" + image_data)
print("Shape images:", images.shape)

In [ ]:
images

In [ ]:
cd=images[[True if i in ["cat", "dog"] else False for i in C]]

In [ ]:
cd.shape

normalizing the data from 8 bit per channel $ 2^8 $ colors to float between 0 and 1.

In [ ]:
gc.collect()
cd=cd/255

In [ ]:
cd

taking a look at an image

In [ ]:
gc.collect()
n=int(cd.shape[0]/2-10000)
n=int(cd.shape[0]-10000)
Image.fromarray(giveIMG(cd[n]))

________________________________________________

# Building a CNN Classifier

In [ ]:
def CLASSIFIER(inpShape: int, do=0.5):
    
    inp = Input(shape = (inpShape,), name = "image_C")
    x = Reshape((64,64,3))(inp)

    x = Conv2D(
        32, 
        kernel_size=(6,6), 
        strides=(2,2), 
        activation='relu',
        name="Dconv1"
    )(x)
    
    x = Conv2D(
        64, 
        kernel_size=(3,3), 
        strides=(2, 2), 
        activation='relu', 
        name="Dconv2"
    )(x)
    x = Conv2D(
        128,
        kernel_size=(2, 2), 
        strides=(1, 1), 
        activation='relu', 
        name="Dconv3"
    )(x)
    
    x = Flatten()(x)
    x = Dropout(do)(x)
    
    out = Dense(1, activation='sigmoid')(x)
    
    m = Model(inputs = inp, outputs = out, name="Classifier")
    
    return m

In [ ]:
int(np.prod(list((64,64,3))))

In [ ]:
loss = BinaryCrossentropy(label_smoothing=0.2)

In [ ]:
Cls = CLASSIFIER(inpShape = 12288)
Cls.compile(optimizer=Adam(0.001), loss=loss, metrics=["BinaryAccuracy"])
print("Classifier trainable:", Cls.trainable)
Cls.summary()

# Training the CNN

In [ ]:
gc.collect()
folds=3
rs = KFold(n_splits=folds, shuffle=True, random_state=123)
for tr, te in rs.split(cd):
    history = History()

history = Cls.fit(
    x=tf.convert_to_tensor(cd[tr]), 
    y=target[tr],
    validation_data=(cd[te], target[te]),
    epochs = 10, 
    batch_size = 48, 
    shuffle = True,
    verbose=1
    )

In [ ]:
def imageBig(image):
    img = Image.fromarray(giveIMG(image))
    plt.imshow(img)
    plt.show()

for im in [11, 12]:
 imageBig(cd[im])

In [ ]:
prediction = Cls.predict(
   x=tf.convert_to_tensor(cd[[11,12]]), 
   batch_size = 1
)

In [ ]:
prediction

OK, both are Dogs, now lets attack!

# Shap

Training the gradient explainer.

In [ ]:
gc.collect()
# select a set of background examples to take an expectation over
background = cd[np.random.choice(cd.shape[0], 1000, replace=False)]

# explain predictions of the model on three images
e = shap.GradientExplainer(Cls, background)

### Dogs

We define 100 examples of cat and dog images each for further analysis.

In [ ]:
samps=100

In [ ]:
shap_values = e.shap_values(cd[0:samps])

In [ ]:
viz_shap_values = shap_values.reshape(-1,64,64,3)

In [ ]:
viz_cd = cd[0:samps].reshape(-1,64,64,3)

### Cats

In [ ]:
samps=100

In [ ]:
single_shap = e.shap_values(cd[-samps:])
single_shap.shape

In [ ]:
viz_single_shap = single_shap.reshape(samps,64,64,3)
viz_single_shap.shape

In [ ]:
viz_cdct = cd[-samps:].reshape(samps,64,64,3)
viz_cdct.shape

In [ ]:
shap.image_plot(viz_single_shap[:3,:,:,:], viz_cdct[:3,:,:,:], show=False)
plt.savefig("shap1_3x.jpeg")

One of the key findings is that the relationship between Shap Value and Pixel Value is not always positive, as one might expect, but is sometimes positive, sometimes negative, and mostly neutral. This symmetry results in a butterfly pattern, which is exploited in the attack.

In [ ]:
chose1=239
chose2=116

fig, ax = plt.subplots(1,2, figsize=(13,3*2), sharey=True)
for i, chose in enumerate([chose1, chose2]):
    ax[i].scatter(cd[-samps:].reshape(100,64*64*3,1)[:,chose,0], single_shap[:,chose,0], c="black")
    if i == 0:
        ax[i].set_ylabel("SHAP Value $\phi$")
    ax[i].set_xlabel("Pixel-Value")
    ax[i].set_title("{} Pixel".format(["A", "Another"][i]), fontsize=25)
fig.tight_layout(pad=1)
plt.savefig("shap_bspl1_phi.jpeg")

Here, let's just take a look at a whole bunch of images with all the pixels layered on top of each other. 

In [ ]:
alpha=0.3
box_col="grey"
t_shap=0.003
t_x=0.25
class_names = ["dogs", "cats"]

fig, ax = plt.subplots(1,2, figsize=(13,3*2), sharey=True)
imgs_s = [cd[0:100].reshape(samps,64*64*3,1), cd[-samps:].reshape(100,64*64*3,1)]
for i, s in enumerate([shap_values, single_shap]):
    ax[i].fill_between(x=[1-t_x,1], y1=0.05, y2=t_shap, color=box_col, alpha=alpha)
    ax[i].fill_between(x=[0,t_x], y1=0.05, y2=t_shap, color=box_col, alpha=alpha)
    ax[i].fill_between(x=[1-t_x,1], y1=-t_shap, y2=-0.05, color=box_col, alpha=alpha)
    ax[i].fill_between(x=[0,t_x], y1=-t_shap, y2=-0.05, color=box_col, alpha=alpha)
    ax[i].scatter(imgs_s[i], s, alpha=0.2, c="black")
    
    ax[i].hlines(xmin=0, xmax=t_x,  y=t_shap, color='w', linestyle='--', linewidth=2,)
    ax[i].vlines(ymin=t_shap, ymax=0.04, x=t_x, color='w', linestyle='--', linewidth=2,)
    ax[i].hlines(xmin=0, xmax=t_x,  y=-t_shap, color='w', linestyle='--', linewidth=2,)
    ax[i].vlines(ymin=-0.03, ymax=-t_shap, x=t_x, color='w', linestyle='--', linewidth=2,)    
    ax[i].hlines(xmin=1-t_x, xmax=1,  y=t_shap, color='w', linestyle='--', linewidth=2,)
    ax[i].vlines(ymin=t_shap, ymax=0.04, x=1-t_x, color='w', linestyle='--', linewidth=2,)
    ax[i].hlines(xmin=1-t_x, xmax=1,  y=-t_shap, color='w', linestyle='--', linewidth=2,)
    ax[i].vlines(ymin=-0.03, ymax=-t_shap, x=1-t_x, color='w', linestyle='--', linewidth=2,)

    ax[i].set_ylim([-0.01, 0.01])
    ax[i].set_xlim([0,1])
    if i == 0:
        ax[i].set_ylabel("SHAP Value $\phi$")
    ax[i].set_xlabel("Pixel-Value")
    ax[i].set_title("Class {}".format(class_names[i]), fontsize=25)
fig.tight_layout(pad=1)
plt.savefig("shap2a_phi.jpeg")
plt.show()        

The gray areas are neutralized. You can see that this applies to all images.

t_shap is recommended to be std of shap values

In [ ]:
np.std(shap_values)*2 #t_shap

In [ ]:
def shapAttack(x, shapVal, strength=60, t_shap=0.001, t_x=0.0, mid_clip=0.1, view=True, up=True):
    """
    t_shap is recommended to be std of shap values
    """
    t_shap = t_shap*strength/10#??
    attack_vector = shapVal*strength
    
    #right
    n_upper = np.where((attack_vector > t_shap) & (x > 0.5 + t_x), -attack_vector, 0)
    right = np.where((n_upper != 0) & (x + n_upper < 0.5-mid_clip), x - 0.5-mid_clip, n_upper)
    if up:
        n_low = np.where((attack_vector < -t_shap) & (x > 0.5 + t_x), -attack_vector, 0)
        n_low = np.where((n_low != 0) & (x + n_low < 0.5-mid_clip), x - 0.5-mid_clip, n_low)
        right = right + n_low
    
    #left
    p_low = np.where((attack_vector > t_shap) & (x < 0.5 - t_x), attack_vector, 0)
    left = np.where((p_low != 0) & (x + p_low > 0.5+mid_clip), 0.5+mid_clip-x, p_low)
    if up:   
        p_upper = np.where((attack_vector < -t_shap) & (x < 0.5 - t_x), attack_vector, 0)
        p_upper = np.where((p_upper != 0) & (x + p_upper > 0.5+mid_clip), 0.5+mid_clip-x, p_upper)
        left = left + p_upper
    
    attack_vector = right + left
    
    #visualization
    if view:
        s_max = (attack_vector+np.abs(attack_vector.min()))
        s = s_max/s_max.max()*255
        img = giveIMG(s)
        plt.imshow(img)
        plt.show()
    return attack_vector

# Multiple Attack

Attecking 100 images to check the performance.

In [ ]:
misClas=0
smaller=0

for i_, (s, x) in enumerate(zip(single_shap, cd[-samps:])):
    at = shapAttack(x, s.T, t_shap=np.std(single_shap)*2, view=False, up=True)
    attacked = np.clip(x + at, 0, 1)
    prediction = Cls.predict(
       x=np.array([attacked.reshape(64*64*3)]), 
       batch_size = 1, 
       verbose=False
    )
    prediction_o = Cls.predict(
       x=np.array([x.reshape(64*64*3)]), 
       batch_size = 1, 
       verbose=False
    )
    
    if prediction[0] <= 0.5:
        misClas+=1
    
    if prediction[0] <= prediction_o[0]:
        smaller+=1
    
    print("Attacked:", prediction[0], "Original:", prediction_o[0], "Delta:", prediction_o[0]-prediction[0], "iter:", -100+i_)
print(misClas, "% misclassified")

In [ ]:
print(smaller, "images showed a decrease in the accuracy of classifying the correct class.")

Here’s another quick comparison between the originals and the doctored images. As you can see, it’s not just the machine that’s being fooled, but human perception as well, since the images are nearly identical.

In [ ]:
samples = [-100,-1, -94, -92,]
fig, ax = plt.subplots(2,len(samples), figsize=(13,3*2), sharey=True)

for j in range(2):
    for i, s in enumerate(samples):
        if j == 1:
            at = shapAttack(cd[s], single_shap[s].T, t_shap=np.std(single_shap)*2, view=False, strength=50)
            attacked = np.clip(cd[s] + at, 0,1)
        else:
            attacked = cd[s]
        prediction = Cls.predict(
           x=np.array([attacked.reshape(64*64*3)]), 
           batch_size = 1,
            verbose=False
        )
        prediction = round(prediction[0][0] * 100)
        img = Image.fromarray((attacked*255).astype('uint8').reshape((64,64,3)))
        ax[j, i].axis('off')
        ax[j, i].imshow(img)
        ax[j, i].set_title(f"prob cat = {prediction}%", fontsize=15)
plt.savefig("shap_comp.jpeg")
plt.show()

Here’s another quick look. We’ll show you exactly how an image is distorted by adding noise.

In [ ]:
selected = -97 #we take image -97 from 100

In [ ]:
at = shapAttack(cd[selected], single_shap[selected].T, t_shap=np.std(single_shap)*2)
at.shape

so, this pertubation is added to the original

In [ ]:
attacked = np.clip(cd[selected] + at, 0,1)

In [ ]:
imageBig(attacked.reshape(64,64,3))

In [ ]:
prediction = Cls.predict(
   x=np.array([attacked.reshape(64*64*3)]), 
   batch_size = 1
)
print("probability of a cat", round(prediction[0][0], 2))

```
@misc{mollard2026adversarial,
  title         = {Adversarial Evasion Attacks on Computer Vision using SHAP Values},
  author        = {Frank Mollard and Marcus Becker and Florian Roehrbein},
  year          = {2026},
  eprint        = {2601.10587},
  archivePrefix = {arXiv},
  primaryClass  = {cs.CV},
  doi           = {10.48550/arXiv.2601.10587}
}
```